# Massive Fine-Tuning of GuardRail AI
This notebook merges your local CSV datasets with two Hugging Face datasets and fine-tunes DeBERTa-v3 on a T4/A100 GPU.

**Instructions:**
1. Go to `Runtime > Change runtime type` and select **T4 GPU**.
2. Run the cells sequentially.

In [ ]:
# 1. Install dependencies
!pip install transformers[torch] datasets accelerate pandas scikit-learn

In [ ]:
# 2. Mount Google Drive
# This will ask you to login. It allows us to read your CSVs and save the final model.
from google.colab import drive
drive.mount('/content/drive')

### ⚠️ Action Required ⚠️
Please upload your two CSV files:
- `combined_prompt_injection_threat_matrix_dataset.csv`
- `prompt_injection_threat_matrix.csv`

Upload them into a folder in your Google Drive, for example: `My Drive/GuardRail_Datasets/`.
Update the paths below to match exactly where you uploaded them.

In [ ]:
import pandas as pd
from datasets import load_dataset, Dataset, concatenate_datasets, Value

# --- UPDATE THESE PATHS --- #
local_csv_1 = '/content/drive/MyDrive/GuardRail_Datasets/combined_prompt_injection_threat_matrix_dataset.csv'
local_csv_2 = '/content/drive/MyDrive/GuardRail_Datasets/prompt_injection_threat_matrix.csv'

# 1. Load Local Datasets
df1 = pd.read_csv(local_csv_1)[['text', 'label']].dropna()
df2 = pd.read_csv(local_csv_2)[['text', 'label']].dropna()

ds_local1 = Dataset.from_pandas(df1).cast_column('label', Value('int64'))
ds_local2 = Dataset.from_pandas(df2).cast_column('label', Value('int64'))

print(f"Local DS 1: {len(ds_local1)} rows")
print(f"Local DS 2: {len(ds_local2)} rows")

# 2. Load Hugging Face Dataset 1
print("Loading neuralchemy/Prompt-injection-dataset...")
hf1 = load_dataset('neuralchemy/Prompt-injection-dataset', split='train')
# Keep only 'text' and 'label'
hf1 = hf1.remove_columns([c for c in hf1.column_names if c not in ['text', 'label']])
hf1 = hf1.cast_column('label', Value('int64'))

# 3. Load Hugging Face Dataset 2
print("Loading reshabhs/SPML_Chatbot_Prompt_Injection...")
hf2 = load_dataset('reshabhs/SPML_Chatbot_Prompt_Injection', split='train')
# Rename columns to match
hf2 = hf2.rename_column('User Prompt', 'text')
hf2 = hf2.rename_column('Prompt injection', 'label')
hf2 = hf2.remove_columns([c for c in hf2.column_names if c not in ['text', 'label']])
hf2 = hf2.cast_column('label', Value('int64'))

# 4. Combine Everything
final_dataset = concatenate_datasets([ds_local1, ds_local2, hf1, hf2])
final_dataset = final_dataset.shuffle(seed=42)

print(f"\nTotal Combined Dataset Size: {len(final_dataset)} rows!")
print(final_dataset[0])

In [ ]:
# 3. Tokenize Data
from transformers import AutoTokenizer

MODEL_NAME = "ProtectAI/deberta-v3-base-prompt-injection-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    # Convert to strings in case of bad data types
    texts = [str(x) for x in examples["text"]]
    return tokenizer(texts, padding="max_length", truncation=True, max_length=128)

print("Tokenizing massive dataset...")
tokenized_dataset = final_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

# Ensure labels are integers
def cast_labels(example):
    example["label"] = int(example["label"])
    return example
tokenized_dataset = tokenized_dataset.map(cast_labels)

In [ ]:
# 4. Train Model on GPU
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

OUTPUT_DIR = "/content/drive/MyDrive/GuardRail_FineTuned"

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,   # T4 GPU can handle 16 easily
    gradient_accumulation_steps=2,    # Effective batch size of 32
    fp16=True,                        # Mixed precision for massive speedup
    learning_rate=2e-5,
    save_strategy="epoch",
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

print("🚀 Starting Training on Colab GPU...")
trainer.train()

In [ ]:
# 5. Save Final Model to Google Drive
print(f"💾 Saving final model to {OUTPUT_DIR} ...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ Done! You can now download this folder from your Google Drive.")